# Snowpark Data Processing Example

This notebook demonstrates file path patterns in Snowpark operations.

In [ ]:
from snowflake.snowpark import Session
import pandas as pd

# Initialize session
session = Session.builder.configs(connection_params).create()

## Load Data from S3

In [ ]:
# Read customer data from S3
customers = session.read.parquet("s3://company-data/customers/customer_master.parquet")

# Read sales transactions
sales = session.read.csv("s3://company-data/sales/transactions.csv")

## Process and Transform

In [ ]:
# Join and aggregate
result = customers.join(sales, "customer_id").group_by("region").agg({"amount": "sum"})

# Write results back to HDFS
result.write.parquet("hdfs://prod-cluster/warehouse/sales_by_region.parquet")

## Load Reference Data

In [ ]:
# Load configuration
config_path = "./config/processing_params.json"
config = pd.read_json(config_path)

# Load lookup tables from relative paths
regions = pd.read_csv("../reference/regions.csv")
categories = pd.read_csv("../reference/product_categories.csv")

## Dynamic Paths

In [ ]:
# Use environment-specific paths
env = "production"
bucket = "data-lake"

input_path = f"s3://{bucket}-{env}/raw/daily_data.parquet"
output_path = f"hdfs://cluster/processed/{env}/results.parquet"

data = session.read.parquet(input_path)
processed = data.filter(data["status"] == "active")
processed.write.parquet(output_path)

## Already Converted Stage Paths

These paths have already been converted by SMA to use stage format.

In [ ]:
# Existing stage paths (prefix will be updated)
archive = session.read.parquet("@OLD_STAGE/s3/archive-bucket/historical_data.parquet")
reference = session.read.csv("@OLD_STAGE/hdfs/cluster/reference/master_data.csv")